# Introduction

In this module, we will take a sample image from ICEYE and run produce a binary water map. We will do this by following these steps

1. Obtain ICEYE image
2. Speckle Filter the image
3. Psuedo Terrain correct the image
4. Determine initial threshold through sampling points that have recurring water using JRC
5. Run HYDRAFloods Edge Otsu algorithm

# Part 1: Housekeeping and obtaining ICEYE image

In [ ]:
my_gee_folder = 'users/mickymags/watchduty/'

In [ ]:
!pip install ipyleaflet==0.18.2 geemap hydrafloods     # Install hydrafloods and its relevant dependencies
!pip install geemap

In [ ]:
from hydrafloods import corrections
import hydrafloods as hf
import ee
import geemap
import matplotlib.pyplot as plt

In [ ]:
ee.Authenticate()
ee.Initialize(project = 'servir-sco-assets')

In [ ]:
imgcollection = ee.ImageCollection('users/mickymags/watchduty/alaska_iceye')
ic1 = imgcollection.filterDate('2025-10-11', '2025-10-13').first();
ic2 = imgcollection.filterDate('2025-10-13', '2025-10-15').first();
#ic3 = imgcollection.filterDate('2025-10-16', '2025-10-18').first();

ic1_proj = ic1.projection()
ic1_scale = ic1_proj.nominalScale().getInfo()

ic2_proj = ic2.projection()
ic2_scale = ic2_proj.nominalScale().getInfo()

#ic3_proj = ic3.projection()
#ic3_scale = ic3_proj.nominalScale().getInfo()

print('Image 1 Projection: {0:0.4f}'.format(ic1_scale))
print('Image 2 Projection: {0:0.4f}'.format(ic2_scale))
#print('Image 3 Projection: {0:0.4f}'.format(ic3_scale))

In [ ]:
#aoi = ee.FeatureCollection('users/mickymags/iceye_aoi')
aoi = ee.FeatureCollection('users/mickymags/watchduty/ak_aoi1')

#aoi = ee.FeatureCollection('projects/servir-sco-assets/assets/watchduty/memphis_aoi')
aoi_geom = aoi.geometry()

# Get the coordinates of the center of the AOI for mapping purposes
aoi_centroid = aoi.geometry().centroid()             # Get the center of the AOI
lon = aoi_centroid.coordinates().get(0).getInfo()    # Extract the longitude from the centroid
lat = aoi_centroid.coordinates().get(1).getInfo()    # Extract the latitude from the centroid

In [ ]:
#iceye = hf.Dataset(
#    region = aoi_geom,               #my_geom
    #start_time = '2025-10-11',
#    #end_time = '2025-10-13',
#    asset_id = 'users/mickymags/watchduty/alaska_iceye_v2'
#)

iceye = hf.Dataset(
    region=aoi_geom,
    start_time = '2025-10-13',
    end_time = '2025-10-15',
    asset_id = 'users/mickymags/watchduty/alaska_iceye'
)

test histrogram

In [ ]:
# Load your SAR image from assets
asset_id1 = "users/mickymags/watchduty/alaska_iceye/closeup"
asset_id2 = "users/mickymags/watchduty/alaska_iceye/closeup2"
image = ee.Image(asset_id1)
image = image.rename('VV', 'angle')

image2 = ee.Image(asset_id2)
image2 = image2.rename('VV', 'angle')

# Get band names
bands = image.select('VV').bandNames().getInfo()
print("Bands in image:", bands)

# Create a histogram for each band
def plot_histograms(image, bands,  aoi_descriptor, region, xmin, xmax, scale=3.46325, n_bins=4096):
    """Plot histograms for each band of an ee.Image."""
    fig, axes = plt.subplots(len(bands), 1, figsize=(8, 4*len(bands)))

    if len(bands) == 1:
        axes = [axes]

    for i, band in enumerate(bands):
        # Reduce region to histogram
        hist_dict = image.select(band).reduceRegion(
            reducer=ee.Reducer.histogram(maxBuckets=n_bins), #maxBuckets=n_bins),
            geometry=region,
            scale=scale,

            bestEffort=True
        ).get(band).getInfo()

        if hist_dict is not None:
            counts = hist_dict['histogram']
            bucketMeans = hist_dict['bucketMeans']

            axes[i].bar(bucketMeans, counts, width=(bucketMeans[1]-bucketMeans[0]))
            axes[i].set_title(f"Histogram of {band} for {aoi_descriptor}")
            axes[i].set_xlabel("Pixel values")
            axes[i].set_ylabel("Frequency")
            axes[i].set_xlim(xmin, xmax) #0, 250
        else:
            axes[i].set_title(f"{band}: No data")

    plt.grid()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_histograms(image, bands, 'ICEYE Image 1 (10/12)', aoi_geom, 0, 0.0035, scale = ic1_scale)

In [ ]:
plot_histograms(image2, bands, 'ICEYE Image 2 (10/14)', aoi_geom, 0, 0.0035, scale = ic2_scale)

As we can see, we have a very unimodal histogram. This does not bode well for our thresholding algorithm, which attempts to seek a minimum between a bimodal histogram by maximizing intraclass variance

In [ ]:
iceye.n_images

In [ ]:
iceye_med = iceye.collection.first().clip(aoi_geom)    # Clipping to AOI Geom is necessary for speckle filtering
iceye_med2 = iceye2.collection.first().clip(aoi_geom)
iceye_vp = {
    'bands': 'VV', #'b1',
    'min': 0,
    'max': 0.0035
}

Get a sentinel-1 image from the same time

In [ ]:
s1 = ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi_geom).filterDate('2025-10-09', '2025-10-11').first().clip(aoi_geom)
s1c = ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi_geom).filterDate('2025-10-19', '2025-10-23').first().clip(aoi_geom)
s1_vp = {
    'bands': 'VV',
    'min': -20,
    'max': 0
}

In [ ]:
plot_histograms(s1, bands, 'Sentinel-1  2025/10/10', aoi_geom, -25, 0, scale= 30, n_bins = 96)

In [ ]:
plot_histograms(s1, bands, 'Sentinel-1  2025/10/22', aoi_geom, -25, 0, scale= 30, n_bins = 96)

In [ ]:
iceye_med_bands = iceye_med.bandNames().getInfo()
iceye_med = iceye_med.rename('VV', 'angle')
iceye_med2 = iceye_med2.rename('VV', 'angle')

iceye_lee = hf.lee_sigma(iceye_med)
iceye_gamma = hf.gamma_map(iceye_med, enl = 4, window = 20)
iceye_refined = hf.refined_lee(iceye_med)

iceye2_lee = hf.lee_sigma(iceye_med2)
iceye2_gamma = hf.gamma_map(iceye_med2)
iceye2_refined = hf.refined_lee(iceye_med2)

In [ ]:
#ic_lee = iceye.apply_func(hf.lee_sigma)
#ic_gamma = iceye.apply_func(hf.gamma_map)
#ic_refined = iceye.apply_func(hf.refined_lee)

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 11)
Map.addLayer(aoi_geom, {}, 'Area of Interest')
#Map.addLayer(s1, s1_vp, 'Sentinel-1')
#Map.addLayer(iceye_med, iceye_vp, 'ICEYE')
Map.addLayer(iceye_med2, iceye_vp, 'ICEYE2')
Map.addLayer(iceye2_lee, iceye_vp, 'ICEYE2 Lee')
Map.addLayer(iceye2_gamma, iceye_vp, 'ICEYE2 Gamma')
Map.addLayer(iceye2_refined, iceye_vp, 'ICEYE2 Refined')

Map.addLayerControl()
Map

# Part 2: Speckle Filtering

In [ ]:
plot_histograms(iceye_med2, bands, 'Raw Image 10/14/2025', aoi_geom, 0, 0.0035)

In [ ]:
plot_histograms(iceye2_lee, bands, 'Memphis Lee Filtered Image', aoi_geom, 0, 0.0035)

In [ ]:
plot_histograms(iceye2_gamma, bands, 'Gamma-Map Filtered Image', aoi_geom, 0, 0.0035, n_bins = 256)

In [ ]:
plot_histograms(iceye2_refined, bands, 'Memphis Refined Lee Filtered Image', aoi_geom, 0, 0.0035)

# Part 3: Psuedo terrain correction

In [ ]:
#elv = ee.Image("JAXA/ALOS/AW3D30/V2_2").select("AVE_DSM")
elv = ee.ImageCollection("USGS/3DEP/10m_collection").filterBounds(aoi_geom).mosaic().clip(aoi_geom)

In [ ]:
print(elv.getInfo())

In [ ]:
#iceye terrain corrected
ic2_tc = corrections.slope_correction(iceye_med2, elevation=elv, scale = 10)#iceye2_gamma, buffer=30)

In [ ]:
ic2_tc.getInfo()

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 12)
Map.addLayer(aoi_geom, {}, 'Area of Interest')
#Map.addLayer(iceye_med2, iceye_vp, 'ICEYE')
#Map.addLayer(iceye2_gamma, iceye_vp, 'ICEYE Speckle')
Map.addLayer(ic2_tc, iceye_vp, 'Speckle + TC')

Map.addLayerControl()
Map

# Part 4: Initial Threshold determination

In [ ]:
iceye_sampled = iceye_med2.sample(aoi, numPixels = 5000, scale = ic2_scale).getInfo()
features = iceye_sampled['features']
first_feature = features[0]
first_feature['properties']['VV']

vals = []

for j in range(len(features)):
  feat_of_int = features[j]
  vals.append(feat_of_int['properties']['VV'])

In [ ]:
import numpy as np

In [ ]:
np_vals = np.array(vals)
np_vals.max()

In [ ]:
np_vals.mean()

# Part 5: Run Edge Otsu Thresholding Algorithm

In [ ]:
ic2_scale

In [ ]:
edge = hf.edge_otsu(
    iceye_med2, #iceye_med
    band = 'VV',
    region = aoi,
    edge_buffer = 30,
    initial_threshold = np_vals.mean(),
    thresh_no_data = 30,
    scale = 10
)

In [ ]:
water_vp = {
    'palette': ['000000', 'add8e6'],
    'min': 0,
    'max': 1
}

In [ ]:
# Edge algorithm using the refined-lee-sigma speckle filtering method
edge_refined = hf.edge_otsu(
    iceye_med2, #iceye_med
    band = 'VV',
    region = aoi_geom,
    edge_buffer = 300,
    initial_threshold = 92,
    thresh_no_data = 90,
    scale = 10
)

In [ ]:
planet = ee.Image('projects/servir-sco-assets/assets/watchduty/memphis_planet')

pl_vp = {
    'bands': ['b3', 'b2', 'b1'],
    'min': 0,
    'max': 3000
}

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 12)
Map.addLayer(aoi, {}, 'Area of Interest')
Map.addLayer(iceye_med2, iceye_vp, 'ICEYE')
#Map.addLayer(ic_lee_med, iceye_vp, 'ICEYE Lee')
#Map.addLayer(ic_refined_med, iceye_vp, 'ICEYE Refined');
Map.addLayer(edge, water_vp, 'Edge')
#Map.addLayer(planet, pl_vp, 'Plantet')
Map.addLayer(edge_refined, water_vp, 'Edge Refined')
#Map.addLayer(bmax, water_vp, 'Bmax')
Map.addLayerControl()
Map

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 12)
Map.addLayer(aoi, {}, 'Area of Interest')
Map.addLayer(iceye_med2, iceye_vp, 'ICEYE')
#Map.addLayer(ic_lee_med, iceye_vp, 'ICEYE Lee')
#Map.addLayer(ic_refined_med, iceye_vp, 'ICEYE Refined');
Map.addLayer(edge_refined, water_vp, 'Edge')
#Map.addLayer(planet, pl_vp, 'Plantet')
#Map.addLayer(edge_refined, water_vp, 'Edge Refined')
#Map.addLayer(bmax, water_vp, 'Bmax')
Map.addLayerControl()
Map

In [ ]:
#

In [ ]:
iceye_scale

In [ ]:
geemap.ee_export_image_to_drive(image = edge_refined,
                                region = export_aoi,
                                folder = 'watchduty',
                                description = 'hf_edge_refined_v2',
                                scale=3.463225,
                                maxPixels = 1e13)

In [ ]:
geemap.ee_export_image_to_drive(image = edge,
                                region = export_aoi,
                                folder = 'watchduty',
                                description = 'hf_test_edge',
                                scale=3.463225,
                                maxPixels = 1e13)

geemap.ee_export_image_to_drive(image = edge,
                                region = export_aoi,
                                folder = 'watchduty',
                                description = 'hf_test_bmax',
                                scale=3.463225,
                                maxPixels = 1e13)


# Part 6: Workflow demonstration

This section is meant to demonstrate the entire HYDRAFloods workflow within one code cell

In [ ]:
# Get ICEYE dataset
iceye = hf.Dataset(
    region = aoi_geom,                                         # The region of interest (of type Earth Engine geometry)
    start_time = '2020-03-30',                                 # The start date of interest
    end_time = '2020-04-01',                                   # The end date of interest
    asset_id = 'projects/my_gee_project/my_iceye_images'       # The GEE Asset ID of the ICEYE imagery
)

# Speckle Filtering
iceye_speckle = iceye.apply_func(hf.lee_sigma)                                  # Apply the Lee Sigma speckle filter to the above dataset
ic_tc = iceye_speckle.apply_func(corrections.slope_correction, elevation=elv,
                                 buffer=30).collection().median()               # Apply a terrain correction
# Histogram Thresholding
edge = hf.edge_otsu(                 # Apply the Edge Otsu algorithm
    ic_tc,                          # Using the speckle-filtered, terrain-corrected Iceye Image
    band = 'VV',                     # On just the Vertically polarized band
    region = aoi_geom,               # Over the aforementioned region of interest
    edge_buffer = 30,                # Buffer the edges identified by the Canny algorithm by 30 meters on either side to sample
    initial_threshold = 40,          # The initial threshold value the edge otsu will try to improve upon
    thresh_no_data = 30,             # The threshold to use for pixels masked by terrain correction
    scale = 3                        # The scale in meters to use
)